draw EECs and ratios for injected v2 plots

In [1]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import os
import ctypes
import math
import array

In [2]:
high_bins = [ [60,71], [71,78], [78,91], [91,97], [97,1000] ]
inclusive_bins = [ [0,25], [25,36], [36,48], [48,60]  ]
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [3]:
ROOT.gDirectory.Clear()
filepath_0mb_inc = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_inclusive/EEC_non_binned_Output_Batch0.root"
filepath_3mb_inc = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/3mb_inclusive/EEC_non_binned_Output_Batch0.root"

filepath_0mb_01_v2 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_inclusive/injected_v2_0_1/EEC_non_binned_Output_Batch0.root"



In [4]:
# get and normalise the histograms

def initialise(filepath1, filepath2):
    '''
    returns normalised EECs
    '''

    tfile1 = ROOT.TFile.Open(filepath1, "READ")
    num_jets1 = tfile1.Get("num_jets_STD").GetVal()
    EEC1 = tfile1.Get("hEEC_STD")
    EEC1.SetDirectory(0)
    EEC1.Scale(1/num_jets1)
    tfile1.Close()

    tfile2 = ROOT.TFile.Open(filepath2, "READ")
    num_jets2 = tfile2.Get("num_jets_STD").GetVal()
    EEC2 = tfile2.Get("hEEC_STD")
    EEC2.SetDirectory(0)
    EEC2.Scale(1/num_jets2)
    tfile2.Close()

    return EEC1, EEC2

In [6]:
def rebin_log(hist, name, num_bins_target=50):
    '''
    input: 
    - the ratio histogram
    - the name of the new rebinned histograms
    - the desired number of logarithmic bins
    
    output: rebinned log-spaced versions of the histograms
    '''
    import array
    
    # 1. Define the limits (your histograms range from 0 to 1)
    # Since log(0) is undefined, start slightly above 0 matching your plot limits (e.g., 10^-3)
    xmin = 0.001
    xmax = 1.0
    
    # 2. Generate log-spaced bin edges using numpy
    # logspace arguments are the exponents: 10^-3 to 10^0
    bin_edges_np = np.logspace(math.log10(xmin), math.log10(xmax), num_bins_target + 1)
    
    # 3. Convert the numpy array to a standard Python/C-compatible array for ROOT
    bin_edges_array = array.array('d', bin_edges_np)
    
    # 4. Create new empty histograms with variable bin widths
    new_hist = ROOT.TH1D(name, f"{hist.GetTitle()};#Delta R;EEC", num_bins_target, bin_edges_array)
    
    # Ensure Sumw2 is enabled to compute correct errors for variable bin widths
    new_hist.Sumw2()
    
    # 5. Manually remap the bin contents from the old fine linear bins to the new log bins
    # We loop over the fine uniform bins of the original histogram
    for old_bin in range(1, hist.GetNbinsX() + 1):
        bin_center = hist.GetBinCenter(old_bin)
        
        # Skip values below our log starting threshold to prevent underflow clutter
        if bin_center < xmin:
            continue
            
        bin_content = hist.GetBinContent(old_bin)
        bin_error = hist.GetBinError(old_bin)
        
        # Find which log-bin this center falls into
        new_bin = new_hist.FindBin(bin_center)
        
        # Add content and propagate errors in quadrature
        new_hist.SetBinContent(new_bin, new_hist.GetBinContent(new_bin) + bin_content)
        new_hist.SetBinError(new_bin, math.sqrt(new_hist.GetBinError(new_bin)**2 + bin_error**2))
        
    return new_hist

In [5]:
colours = [ROOT.kBlue+1, ROOT.kMagenta+1, ROOT.kSpring+4, ROOT.kRed+1, ROOT.kOrange+7]

# inclusive

## EEC plots

In [13]:
# log version, with markers

ROOT.gDirectory.Clear()

eec_0mb , eec_3mb = initialise(filepath_0mb_inc, filepath_3mb_inc)
dummy , eec_01_v2 = initialise(filepath_0mb_inc, filepath_0mb_01_v2)

# make TGraphErrors
tgraph_eec_0mb = ROOT.TGraphErrors(eec_0mb)
tgraph_eec_3mb = ROOT.TGraphErrors(eec_3mb)
tgraph_eec_v2 = ROOT.TGraphErrors(eec_01_v2)

# Choosing colours:
tgraph_eec_0mb.SetLineColorAlpha(colours[0], 0.5)
tgraph_eec_3mb.SetLineColorAlpha(colours[4], 0.5)
tgraph_eec_v2.SetLineColorAlpha(colours[2], 0.5)

tgraph_eec_0mb.SetMarkerColorAlpha(colours[0], 0.5)
tgraph_eec_3mb.SetMarkerColorAlpha(colours[4], 0.5)
tgraph_eec_v2.SetMarkerColorAlpha(colours[2], 0.5)

# markers:
tgraph_eec_0mb.SetMarkerStyle(20)
tgraph_eec_3mb.SetMarkerStyle(20)
tgraph_eec_v2.SetMarkerStyle(20)

tgraph_eec_0mb.SetMarkerSize(0.4)
tgraph_eec_3mb.SetMarkerSize(0.4)
tgraph_eec_v2.SetMarkerSize(0.4)

# initialise canvas
canvas = ROOT.TCanvas("c_eec_l", "log EECs for inclusive datasets", 750, 650)

canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg = ROOT.TMultiGraph()

# Add the data markers
mg.Add(tgraph_eec_0mb, "PE")
mg.Add(tgraph_eec_3mb, "PE")
mg.Add(tgraph_eec_v2, "PE")

# title
mg.SetTitle("EEC for inclusive datasets; #Delta R_{L}; E2C")
mg.Draw("A") 

# Log scale axes
canvas.SetLogx(1)
canvas.SetLogy(1)
mg.GetXaxis().SetLimits(0.001, 1.0)
mg.GetYaxis().SetLimits(0.0001, 1.0)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Legend setup
legend = ROOT.TLegend(0.28, 0.28, 0.68, 0.48) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_eec_0mb, "0mb", "PE")
legend.AddEntry(tgraph_eec_3mb, "3.0mb", "PE")
legend.AddEntry(tgraph_eec_v2, "v_{2}=0.1", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_inclusive_log_plot_markers_v2.pdf")
canvas.Close()


Info in <TCanvas::Print>: pdf file EEC_inclusive_log_plot_markers_v2.pdf has been created


## ratios

In [11]:
# ratio, with markers
# log x axis, rebinned
# plot injected over 0mb and 3mb over 0mb ratios


ROOT.gDirectory.Clear()

eec_0mb , eec_3mb = initialise(filepath_0mb_inc, filepath_3mb_inc)

dummy , eec_01_v2 = initialise(filepath_0mb_inc, filepath_0mb_01_v2)  

#rebin
rebinned_eec_0mb = rebin_log(eec_0mb, "eec_0mb_log", 50)
rebinned_eec_3mb = rebin_log(eec_3mb, "eec_3mb_log", 50)
rebinned_eec_01_v2 = rebin_log(eec_01_v2, "eec_01_v2_log", 50)

# 3. Create a clone of the rebinned 3mb histogram to hold the ratio
rebinned_ratio_eecs = rebinned_eec_3mb.Clone("rebinned_ratio_log")
rebinned_v2_eecs = rebinned_eec_01_v2.Clone("rebinned_ratio_v2")

# 4. Divide the rebinned histograms 
rebinned_ratio_eecs.Divide(rebinned_eec_0mb)
rebinned_v2_eecs.Divide(rebinned_eec_0mb)

# make TGraphErrors
tgraph_ratio_3mb = ROOT.TGraphErrors(rebinned_ratio_eecs)
tgraph_ratio_v2 = ROOT.TGraphErrors(rebinned_v2_eecs)

# Choosing colours:
tgraph_ratio_3mb.SetLineColorAlpha(colours[0], 0.6)
tgraph_ratio_v2.SetLineColorAlpha(colours[3], 0.6)

tgraph_ratio_3mb.SetMarkerColorAlpha(colours[0], 0.7)
tgraph_ratio_v2.SetMarkerColorAlpha(colours[3], 0.7)

# markers:
tgraph_ratio_3mb.SetMarkerStyle(20)
tgraph_ratio_v2.SetMarkerStyle(20)

tgraph_ratio_3mb.SetMarkerSize(0.8)
tgraph_ratio_v2.SetMarkerSize(0.8)

# initialise canvas
canvas = ROOT.TCanvas("c_eec_l_r", "rebinned log ratio of EECs for inclusive datasets", 750, 650)

canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg = ROOT.TMultiGraph()

# Add the data markers
mg.Add(tgraph_ratio_3mb, "PE")
mg.Add(tgraph_ratio_v2, "PE")

# title
mg.SetTitle("Ratio of EECs for inclusive datasets; #Delta R_{L}; Ratio")
mg.Draw("A") 

# reference line
line = ROOT.TLine(0, 1.0, 1.0, 1.0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# scale axes
canvas.SetLogx(1)
mg.GetXaxis().SetLimits(0.001, 1.0)
mg.GetYaxis().SetLimits(0, 2)
mg.SetMinimum(0.97)
mg.SetMaximum(1.03)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)


# Legend setup
#legend = ROOT.TLegend(0.28, 0.78, 0.48, 0.88) 
legend = ROOT.TLegend(0.58, 0.28, 0.78, 0.38) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_ratio_3mb, "E2C_{3.0mb} / E2C_{0mb}", "PE")
legend.AddEntry(tgraph_ratio_v2, "E2C_{v_{2}=0.1} / E2C_{0mb}", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_rebinned_inclusive_ratio_log_markers_v2.pdf")
canvas.Close()


Info in <TCanvas::Print>: pdf file EEC_rebinned_inclusive_ratio_log_markers_v2.pdf has been created


In [14]:
# ratio, with markers
# log x axis, original bins
# plot injected over 0mb and 3mb over 0mb ratios


ROOT.gDirectory.Clear()

eec_0mb , eec_3mb = initialise(filepath_0mb_inc, filepath_3mb_inc)

dummy , eec_01_v2 = initialise(filepath_0mb_inc, filepath_0mb_01_v2)  

# 3. Create a clone of the 3mb histogram to hold the ratio
ratio_3mb_eecs = eec_3mb.Clone("rebinned_ratio_log")
ratio_v2_eecs = eec_01_v2.Clone("rebinned_ratio_v2")

# 4. Divide the rebinned histograms 
ratio_3mb_eecs.Divide(eec_0mb)
ratio_v2_eecs.Divide(eec_0mb)

# make TGraphErrors
tgraph_ratio_3mb = ROOT.TGraphErrors(ratio_3mb_eecs)
tgraph_ratio_v2 = ROOT.TGraphErrors(ratio_v2_eecs)

# Choosing colours:
tgraph_ratio_3mb.SetLineColorAlpha(colours[0], 0.6)
tgraph_ratio_v2.SetLineColorAlpha(colours[3], 0.6)

tgraph_ratio_3mb.SetMarkerColorAlpha(colours[0], 0.6)
tgraph_ratio_v2.SetMarkerColorAlpha(colours[3], 0.6)

# markers:
tgraph_ratio_3mb.SetMarkerStyle(20)
tgraph_ratio_v2.SetMarkerStyle(20)

tgraph_ratio_3mb.SetMarkerSize(0.8)
tgraph_ratio_v2.SetMarkerSize(0.8)

# initialise canvas
canvas = ROOT.TCanvas("c_eec_l_r", "log ratio of EECs for inclusive datasets", 750, 650)

canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg = ROOT.TMultiGraph()

# Add the data markers
mg.Add(tgraph_ratio_3mb, "PE")
mg.Add(tgraph_ratio_v2, "PE")

# title
mg.SetTitle("Ratio of EECs for inclusive datasets; #Delta R_{L}; Ratio")
mg.Draw("A") 

# reference line
line = ROOT.TLine(0, 1.0, 1.0, 1.0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# scale axes
canvas.SetLogx(1)
mg.GetXaxis().SetLimits(0.001, 1.0)
mg.GetYaxis().SetLimits(0, 2)
mg.SetMinimum(0.97)
mg.SetMaximum(1.03)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)


# Legend setup
#legend = ROOT.TLegend(0.28, 0.78, 0.48, 0.88) 
legend = ROOT.TLegend(0.58, 0.28, 0.78, 0.38) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_ratio_3mb, "E2C_{3.0mb} / E2C_{0mb}", "PE")
legend.AddEntry(tgraph_ratio_v2, "E2C_{v_{2}=0.1} / E2C_{0mb}", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_inclusive_ratio_log_markers_v2.pdf")
canvas.Close()


Info in <TCanvas::Print>: pdf file EEC_inclusive_ratio_log_markers_v2.pdf has been created
